# **NewsGenie — Agentic AI News & Information Assistant**
### Course End Project (CEP) — Applied Generative AI Specialisation

---

**Overview:**
NewsGenie is a truly agentic AI news assistant built with **LangGraph `create_react_agent`**.
Unlike a deterministic pipeline, the LLM **autonomously decides which tools to invoke**
and in what sequence — a classic ReAct (Reasoning + Acting) loop.

**Agentic Architecture:**
```
User Query → create_react_agent (ReAct loop)
                  ↓ LLM reasons about the query
                  ↓ Selects one or more tools
    ┌─────────────────────────────────────┐
    │  get_top_headlines(category)         │  ← news headlines by category
    │  search_news(query)                  │  ← topic-specific news search
    │  search_web(query)                   │  ← live web facts / real-time data
    │  get_news_categories()               │  ← list available categories
    └─────────────────────────────────────┘
                  ↓ Tool results fed back to LLM
                  ↓ LLM synthesises final answer
              Final Response
```

**Key components:** `@tool` decorators • `create_react_agent` • `MemorySaver` checkpointer


---
## Part 1: Setup — Environment, Imports & Tool Definitions
---

### **Step 1: Environment Setup**
> `OPENAI_API_KEY` required. `NEWSAPI_KEY` optional — DuckDuckGo is used as fallback.

In [ ]:
import os, warnings, json, time
warnings.filterwarnings("ignore")
from dotenv import load_dotenv
load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY missing from .env"
NEWSAPI_KEY = os.getenv("NEWSAPI_KEY", "")
print("OpenAI key loaded :", bool(os.getenv("OPENAI_API_KEY")))
print("NewsAPI key loaded :", bool(NEWSAPI_KEY and len(NEWSAPI_KEY) > 20))
print("DuckDuckGo fallback: always available (no key needed)")


In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# LangChain / LangGraph
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

print("All imports successful.")


---
### **Step 2: Define Agent Tools with @tool Decorator**

Tools are regular Python functions decorated with `@tool`.
The LLM reads each tool's **docstring** to understand when and how to call it.
This is what makes the system truly agentic — the LLM chooses tools autonomously.


In [ ]:
# ── Helper: category aliases ──────────────────────────────────────────────────
CATEGORY_ALIASES = {
    "tech": "technology", "ai": "technology", "artificial intelligence": "technology",
    "business": "finance", "economy": "finance", "market": "finance",
    "sport": "sports", "soccer": "sports", "football": "sports", "cricket": "sports",
    "medicine": "health", "medical": "health", "wellness": "health",
    "space": "science", "physics": "science", "research": "science",
}
VALID_CATEGORIES = ["technology", "finance", "sports", "health", "science", "general", "entertainment"]

def _normalize_category(category: str) -> str:
    c = category.lower().strip()
    return CATEGORY_ALIASES.get(c, c if c in VALID_CATEGORIES else "general")

def _format_articles(articles: list) -> str:
    if not articles:
        return "No articles found."
    lines = []
    for i, art in enumerate(articles, 1):
        lines.append(f"{i}. **{art['title']}**")
        lines.append(f"   Source: {art.get('source', 'Unknown')}")
        if art.get("description"):
            lines.append(f"   {art['description'][:150]}...")
        if art.get("url"):
            lines.append(f"   URL: {art['url']}")
        lines.append("")
    return "\n".join(lines)

print("Category helpers defined.")


In [ ]:
# ── Tool 1: get_top_headlines ─────────────────────────────────────────────────
@tool
def get_top_headlines(category: str = "general") -> str:
    """
    Fetch the latest top news headlines for a specific category.

    Use this tool when the user asks for news, headlines, or top stories in a category.
    Supported categories: technology, finance, sports, health, science, general, entertainment.

    Examples:
    - "latest technology news" → category="technology"
    - "top sports headlines"   → category="sports"
    - "business news today"    → category="finance"
    """
    cat = _normalize_category(category)

    def _fetch_newsapi():
        if not NEWSAPI_KEY or len(NEWSAPI_KEY) < 20:
            return []
        try:
            cat_map = {"finance": "business"}
            newsapi_cat = cat_map.get(cat, cat)
            url = "https://newsapi.org/v2/top-headlines"
            params = {"apiKey": NEWSAPI_KEY, "language": "en",
                      "pageSize": 5, "category": newsapi_cat}
            resp = requests.get(url, params=params, timeout=10)
            arts = resp.json().get("articles", [])
            return [{"title": a.get("title",""), "source": a.get("source",{}).get("name",""),
                     "description": a.get("description",""), "url": a.get("url","")}
                    for a in arts[:5] if a.get("title")]
        except Exception:
            return []

    def _fetch_ddg():
        try:
            from duckduckgo_search import DDGS
            q = {"technology":"technology news today","finance":"finance business news today",
                 "sports":"sports news today","health":"health medicine news today",
                 "science":"science space news today","general":"top news headlines today"}.get(cat, f"{cat} news today")
            with DDGS() as ddgs:
                raw = list(ddgs.news(q, max_results=5))
            return [{"title": r.get("title",""), "source": r.get("source","DuckDuckGo"),
                     "description": r.get("body",""), "url": r.get("url","")} for r in raw]
        except Exception:
            return []

    articles = _fetch_newsapi() or _fetch_ddg()
    if not articles:
        return f"Could not fetch {cat} news. Please try again."
    return f"## Top {cat.capitalize()} Headlines\n\n" + _format_articles(articles)


print("Tool 1 defined: get_top_headlines")


In [ ]:
# ── Tool 2: search_news ───────────────────────────────────────────────────────
@tool
def search_news(query: str) -> str:
    """
    Search for specific news topics, recent events, or news about a person/company/event.

    Use this tool when the user asks about a specific topic in the news.

    Examples:
    - "latest news about Tesla"           → query="Tesla latest news"
    - "what happened with OpenAI recently" → query="OpenAI recent news"
    """
    def _fetch_newsapi():
        if not NEWSAPI_KEY or len(NEWSAPI_KEY) < 20:
            return []
        try:
            params = {"apiKey": NEWSAPI_KEY, "q": query, "language": "en",
                      "sortBy": "publishedAt", "pageSize": 5}
            resp = requests.get("https://newsapi.org/v2/everything", params=params, timeout=10)
            arts = resp.json().get("articles", [])
            return [{"title": a.get("title",""), "source": a.get("source",{}).get("name",""),
                     "description": a.get("description",""), "url": a.get("url","")}
                    for a in arts[:5] if a.get("title")]
        except Exception:
            return []

    def _fetch_ddg():
        try:
            from duckduckgo_search import DDGS
            with DDGS() as ddgs:
                raw = list(ddgs.news(query, max_results=5))
            return [{"title": r.get("title",""), "source": r.get("source","DuckDuckGo"),
                     "description": r.get("body",""), "url": r.get("url","")} for r in raw]
        except Exception:
            return []

    articles = _fetch_newsapi() or _fetch_ddg()
    if not articles:
        return f"No news found for '{query}'. Try a different search term."
    return f"## News Results: {query}\n\n" + _format_articles(articles)


print("Tool 2 defined: search_news")


In [ ]:
# ── Tool 3: search_web ────────────────────────────────────────────────────────
@tool
def search_web(query: str) -> str:
    """
    Search the web for real-time facts, current data, prices, or general knowledge.

    Use this tool for factual questions that need up-to-date answers:
    - "who is the CEO of Microsoft"
    - "current Bitcoin price"
    - "what is quantum computing"
    """
    try:
        from duckduckgo_search import DDGS
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=4))
        if not results:
            return f"No web results for '{query}'."
        lines = [f"## Web Search Results: {query}\n"]
        for i, r in enumerate(results[:4], 1):
            lines.append(f"{i}. **{r.get('title','')}**")
            lines.append(f"   {r.get('body','')[:200]}...")
            if r.get("href"):
                lines.append(f"   Source: {r['href']}")
            lines.append("")
        return "\n".join(lines)
    except Exception as e:
        return f"Web search unavailable: {e}"


print("Tool 3 defined: search_web")


In [ ]:
# ── Tool 4: get_news_categories ──────────────────────────────────────────────
@tool
def get_news_categories() -> str:
    """
    Return the list of available news categories that NewsGenie can fetch.

    Use this when the user asks what topics or categories are available.
    """
    categories = {
        "technology":    "AI, software, gadgets, cybersecurity, tech companies",
        "finance":       "stocks, markets, economy, business, cryptocurrency",
        "sports":        "football, cricket, tennis, Olympics, sports events",
        "health":        "medicine, wellness, diseases, healthcare, research",
        "science":       "space, physics, climate, discoveries, research",
        "entertainment": "movies, music, celebrities, TV shows, culture",
        "general":       "top stories, breaking news, world events",
    }
    lines = ["## Available News Categories\n"]
    for cat, desc in categories.items():
        lines.append(f"- **{cat.capitalize()}**: {desc}")
    lines.append("\nYou can ask for news in any of these categories!")
    return "\n".join(lines)


TOOLS = [get_top_headlines, search_news, search_web, get_news_categories]
print(f"\nAll {len(TOOLS)} tools defined: {[t.name for t in TOOLS]}")


---
### **Step 3: Create the ReAct Agent with `create_react_agent`**

This is the core of the agentic architecture. Instead of hand-coding routing logic,
we give the LLM a system prompt and a list of tools. The LLM autonomously decides:
1. Which tool(s) to call
2. What arguments to pass
3. How to synthesise the results into a final answer

This is the **ReAct (Reason + Act)** pattern — the agent alternates between
*reasoning* (deciding what to do) and *acting* (calling tools).


In [ ]:
SYSTEM_PROMPT = """You are NewsGenie, an intelligent AI news and information assistant.

Your capabilities:
1. Fetch latest news headlines by category (technology, finance, sports, health, science, entertainment, general)
2. Search for specific news topics or recent events
3. Search the web for real-time facts, data, and information
4. List available news categories

How to respond:
- For news requests (e.g., "latest tech news"), use get_top_headlines with the appropriate category
- For specific topic searches (e.g., "news about Tesla"), use search_news
- For factual/web queries (e.g., "who is the CEO of Apple"), use search_web
- For questions about what's available, use get_news_categories
- Always present results clearly and add a brief insight after news articles
- Remember context from earlier in the conversation for follow-up questions
"""

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3,
    openai_api_key=os.getenv("OPENAI_API_KEY"),
)

memory = MemorySaver()

# ── create_react_agent: the LLM now drives tool selection ─────────────────────
agent = create_react_agent(
    model=llm,
    tools=TOOLS,
    prompt=SYSTEM_PROMPT,
    checkpointer=memory,
)

print("NewsGenie ReAct agent created!")
print(f"LLM       : gpt-4o-mini")
print(f"Tools     : {[t.name for t in TOOLS]}")
print(f"Memory    : MemorySaver (multi-turn conversation)")
print(f"Pattern   : create_react_agent — LLM autonomously selects tools")


---
### **Step 4: Running the Agent — Sample Scenarios**

The `run_agent()` helper invokes the agent and extracts the response
plus which tools the LLM chose to call.


In [ ]:
def run_agent(user_message: str, thread_id: str = "default") -> dict:
    """Run the NewsGenie agent and return response + tools used."""
    config = {"configurable": {"thread_id": thread_id}}
    inputs = {"messages": [{"role": "user", "content": user_message}]}
    result = agent.invoke(inputs, config=config)

    messages     = result.get("messages", [])
    response_text = ""
    tools_used   = []
    tool_inputs  = []

    for msg in messages:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tc in msg.tool_calls:
                tools_used.append(tc["name"])
                tool_inputs.append(tc.get("args", {}))
        elif hasattr(msg, "content") and msg.content and hasattr(msg, "type") and msg.type == "ai":
            response_text = msg.content

    if not response_text and messages:
        last = messages[-1]
        response_text = last.content if hasattr(last, "content") else str(last)

    return {"response": response_text, "tools_used": tools_used, "tool_inputs": tool_inputs}

print("run_agent() helper defined.")


#### Scenario 1: Technology News

In [ ]:
print("=" * 65)
print("SCENARIO 1: Technology Headlines")
print("=" * 65)
r1 = run_agent("What are the latest technology news stories today?", "s1")
print(f"Tools called : {r1['tools_used']}")
print(f"Tool inputs  : {r1['tool_inputs']}")
print()
print(r1["response"][:600])


#### Scenario 2: Finance News

In [ ]:
print("=" * 65)
print("SCENARIO 2: Finance Headlines")
print("=" * 65)
r2 = run_agent("Show me top finance and business headlines", "s2")
print(f"Tools called : {r2['tools_used']}")
print()
print(r2["response"][:600])


#### Scenario 3: Specific News Search

In [ ]:
print("=" * 65)
print("SCENARIO 3: Topic News Search")
print("=" * 65)
r3 = run_agent("What is the latest news about OpenAI?", "s3")
print(f"Tools called : {r3['tools_used']}")
print()
print(r3["response"][:600])


#### Scenario 4: Web Search (Factual Query)

In [ ]:
print("=" * 65)
print("SCENARIO 4: Web Search — CEO Query")
print("=" * 65)
r4 = run_agent("Who is the current CEO of Microsoft?", "s4")
print(f"Tools called : {r4['tools_used']}")
print()
print(r4["response"][:400])


#### Scenario 5: General Knowledge (No Tool Needed)

In [ ]:
print("=" * 65)
print("SCENARIO 5: General Chat — LLM Answers Directly")
print("=" * 65)
r5 = run_agent("What is machine learning and how does it work?", "s5")
print(f"Tools called : {r5['tools_used']} (empty = LLM answered from training data)")
print()
print(r5["response"][:500])


#### Scenario 6: Multi-Turn Memory (Follow-up Questions)

In [ ]:
print("=" * 65)
print("SCENARIO 6: Multi-Turn Memory — Follow-up Questions")
print("=" * 65)
SESSION = "multi_turn_demo"

turn1 = run_agent("Tell me about the latest health news today.", SESSION)
print("Turn 1 tools:", turn1["tools_used"])
print("Turn 1:", turn1["response"][:300], "...\n")

turn2 = run_agent("Which of those stories do you think is most important?", SESSION)
print("Turn 2 tools:", turn2["tools_used"])
print("Turn 2 (follow-up — agent uses memory):")
print(turn2["response"][:400])


#### Scenario 7: Agent Lists Available Categories

In [ ]:
print("=" * 65)
print("SCENARIO 7: List Available Categories")
print("=" * 65)
r7 = run_agent("What news categories can you fetch?", "s7")
print(f"Tools called : {r7['tools_used']}")
print()
print(r7["response"])


---
### **Step 5: Automated Test Cases**

Runs 10 tests validating:
1. Correct tool selection (agent autonomously picks the right tool)
2. Non-empty, relevant responses
3. Multi-turn memory continuity


In [ ]:
test_cases = [
    {"id":"TC-01","query":"What are the latest technology news stories?",
     "expected_tool":"get_top_headlines","check":lambda r,t: "get_top_headlines" in t and len(r)>50},
    {"id":"TC-02","query":"Show me top finance and business headlines",
     "expected_tool":"get_top_headlines","check":lambda r,t: len(r)>50},
    {"id":"TC-03","query":"What are today's top sports stories?",
     "expected_tool":"get_top_headlines","check":lambda r,t: len(r)>50},
    {"id":"TC-04","query":"Latest news in health and medicine",
     "expected_tool":"get_top_headlines","check":lambda r,t: len(r)>50},
    {"id":"TC-05","query":"What is machine learning?",
     "expected_tool":"none","check":lambda r,t: len(r)>100},
    {"id":"TC-06","query":"Who is the current CEO of Microsoft?",
     "expected_tool":"search_web","check":lambda r,t: any(kw in r.lower() for kw in ["satya","nadella","microsoft"])},
    {"id":"TC-07","query":"Search for recent developments in electric vehicles",
     "expected_tool":"search_news","check":lambda r,t: any(kw in r.lower() for kw in ["electric","ev","vehicle","tesla","battery"])},
    {"id":"TC-08","query":"What's happening in science and space today?",
     "expected_tool":"get_top_headlines","check":lambda r,t: len(r)>50},
    {"id":"TC-09","query":"What news categories do you support?",
     "expected_tool":"get_news_categories","check":lambda r,t: "get_news_categories" in t or any(kw in r.lower() for kw in ["technology","finance","sports"])},
    {"id":"TC-10","query":"news",
     "expected_tool":"any","check":lambda r,t: len(r)>50},
]

results_log = []
print(f"Running {len(test_cases)} test cases...\n")

for tc in test_cases:
    try:
        import uuid
        result = run_agent(tc["query"], thread_id=f"tc-{uuid.uuid4()}")
        response = result["response"]
        tools    = result["tools_used"]
        passed   = tc["check"](response, tools)
        status   = "PASS" if passed else "FAIL"
        results_log.append({
            "ID": tc["id"], "Expected Tool": tc["expected_tool"],
            "Tools Used": ", ".join(tools) if tools else "LLM only",
            "Status": status,
            "Response Preview": response[:80]+"..." if len(response)>80 else response
        })
        print(f"  [{status}] {tc['id']} | tools={tools}")
    except Exception as e:
        results_log.append({"ID":tc["id"],"Expected Tool":tc["expected_tool"],
                            "Tools Used":"error","Status":"ERROR",
                            "Response Preview":str(e)[:80]})
        print(f"  [ERROR] {tc['id']}: {e}")
    time.sleep(1)

df = pd.DataFrame(results_log)
passed_n = (df["Status"]=="PASS").sum()
print(f"\nResults: {passed_n}/{len(test_cases)} PASSED ({passed_n/len(test_cases)*100:.0f}%)")


In [ ]:
# Full results table
print("\n=== Test Results ===")
print(df[["ID","Expected Tool","Tools Used","Status"]].to_string(index=False))


---
### **Step 6: Visualisations**


In [ ]:
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi":120,"axes.titlesize":12})


In [ ]:
# ── Chart 1: Test Results + Tool Usage ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

status_counts = df["Status"].value_counts()
colors_s = {"PASS":"#2A9D8F","FAIL":"#E76F51","ERROR":"#E9C46A"}
bar_colors = [colors_s.get(s,"#ccc") for s in status_counts.index]
bars = axes[0].bar(status_counts.index, status_counts.values, color=bar_colors, width=0.4)
axes[0].bar_label(bars, padding=3, fontsize=11)
axes[0].set_title("Test Case Results", fontweight="bold")
axes[0].set_ylabel("Count"); axes[0].set_ylim(0, len(test_cases)+2)

# Tool usage frequency across all test runs
from collections import Counter
all_tools = []
for r in results_log:
    tools_str = r.get("Tools Used","")
    if tools_str and tools_str != "LLM only" and tools_str != "error":
        all_tools.extend([t.strip() for t in tools_str.split(",")])
all_tools.append("LLM only" if not all_tools else "")
tool_counter = Counter(t for t in all_tools if t)
axes[1].bar(tool_counter.keys(), tool_counter.values(),
            color=["#2E86AB","#A23B72","#F18F01","#264653","#2A9D8F"][:len(tool_counter)])
axes[1].set_title("Tool Invocation Frequency", fontweight="bold")
axes[1].set_xlabel("Tool"); axes[1].set_ylabel("Times Called")
plt.xticks(rotation=15)

plt.suptitle("NewsGenie Agent — Test Summary", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("viz_01_test_results.png", bbox_inches="tight")
plt.show()
print("Saved viz_01_test_results.png")


In [ ]:
# ── Chart 2: ReAct Agent Architecture Diagram ─────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 6))
ax.set_xlim(0, 12); ax.set_ylim(0, 7); ax.axis("off")

def box(ax, x, y, w, h, text, color, fontsize=10):
    rect = mpatches.FancyBboxPatch((x-w/2, y-h/2), w, h,
           boxstyle="round,pad=0.15", facecolor=color, edgecolor="#333", linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, y, text, ha="center", va="center", fontsize=fontsize,
            fontweight="bold", color="white", wrap=True)

def arrow(ax, x1, y1, x2, y2, label="", color="#555"):
    ax.annotate("", xy=(x2,y2), xytext=(x1,y1),
                arrowprops=dict(arrowstyle="->", color=color, lw=1.5))
    if label:
        mx, my = (x1+x2)/2, (y1+y2)/2
        ax.text(mx+0.1, my, label, fontsize=8, color=color, style="italic")

# Nodes
box(ax, 6,   6.3, 3.5, 0.8, "User Query", "#2E86AB", 11)
box(ax, 6,   5.0, 4.0, 0.9, "create_react_agent\n(ReAct Loop — LLM decides)", "#A23B72", 10)
box(ax, 6,   3.5, 4.5, 0.8, "LLM Reasoning\n(gpt-4o-mini)", "#264653", 9)

# Tools
box(ax, 2.0, 1.8, 2.8, 0.7, "get_top_headlines\n(category)", "#2A9D8F", 9)
box(ax, 5.2, 1.8, 2.4, 0.7, "search_news\n(query)", "#F18F01", 9)
box(ax, 8.2, 1.8, 2.4, 0.7, "search_web\n(query)", "#E76F51", 9)
box(ax, 11,  1.8, 2.0, 0.7, "get_news\ncategories", "#2E86AB", 9)

box(ax, 6,   0.5, 3.5, 0.7, "Final Response to User", "#2A9D8F", 10)

# Arrows
arrow(ax, 6, 5.9, 6, 5.45)
arrow(ax, 6, 4.55, 6, 3.9)

# LLM → Tools
arrow(ax, 4.5, 3.1, 2.5, 2.15, "tool call", "#2A9D8F")
arrow(ax, 5.5, 3.1, 5.2, 2.15, "tool call", "#F18F01")
arrow(ax, 6.5, 3.1, 7.8, 2.15, "tool call", "#E76F51")
arrow(ax, 7.5, 3.1, 10.5, 2.15, "tool call", "#2E86AB")

# Tools → LLM (results back)
ax.annotate("", xy=(5.0, 3.3), xytext=(2.0, 2.15),
            arrowprops=dict(arrowstyle="->", color="#aaa", lw=1, linestyle="dashed"))
ax.annotate("", xy=(5.5, 3.3), xytext=(5.2, 2.15),
            arrowprops=dict(arrowstyle="->", color="#aaa", lw=1, linestyle="dashed"))

arrow(ax, 6, 3.1, 6, 0.85)

# MemorySaver
mem_rect = mpatches.FancyBboxPatch((9.5, 4.5), 2.2, 0.7,
    boxstyle="round,pad=0.1", facecolor="#E9C46A", edgecolor="#333", linewidth=1)
ax.add_patch(mem_rect)
ax.text(10.6, 4.85, "MemorySaver\n(multi-turn)", ha="center", va="center",
        fontsize=9, fontweight="bold", color="#333")
ax.annotate("", xy=(8, 5.0), xytext=(9.5, 4.85),
            arrowprops=dict(arrowstyle="<->", color="#E9C46A", lw=1.5))

ax.set_title("NewsGenie — Agentic ReAct Architecture", fontsize=13,
             fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig("viz_02_agent_architecture.png", bbox_inches="tight")
plt.show()
print("Saved viz_02_agent_architecture.png")


---
## Conclusion

**NewsGenie** is a truly agentic AI news assistant powered by LangGraph's `create_react_agent`.

### What makes it agentic?

| Aspect | Implementation |
|--------|----------------|
| **Autonomy** | LLM decides which tools to call — no hand-coded routing |
| **ReAct Loop** | Agent alternates reasoning and tool-calling until it has the answer |
| **Tool Use** | `@tool` decorated Python functions with docstrings the LLM reads |
| **Memory** | `MemorySaver` checkpointer enables multi-turn conversation |
| **Adaptability** | Agent can chain multiple tools, or answer without tools if not needed |

### Components

| Component | Technology |
|-----------|------------|
| Agent Framework | `langgraph.prebuilt.create_react_agent` |
| LLM | `gpt-4o-mini` via `langchain_openai.ChatOpenAI` |
| News Source | NewsAPI.org (primary) + DuckDuckGo News (fallback) |
| Web Search | DuckDuckGo text search via `duckduckgo-search` |
| Memory | `langgraph.checkpoint.memory.MemorySaver` |
| UI | Streamlit multi-page app |
| Architecture | Multi-file: `tools.py` → `agents.py` → `workflow.py` → `streamlit_app.py` |

### Key agentic behaviours demonstrated
- ✅ LLM autonomously selects `get_top_headlines` for category news requests
- ✅ LLM autonomously selects `search_news` for specific topic searches
- ✅ LLM autonomously selects `search_web` for factual/live queries
- ✅ LLM answers without tools for general knowledge questions
- ✅ Multi-turn memory via MemorySaver — follow-up questions work correctly
- ✅ DuckDuckGo fallback when NewsAPI key is absent

---
